# STEERLAB COLAB

Chat con **vector steering** su GPU. Nove assi in tre famiglie, barre da un polo
all'altro, equalizzatore per layer.

**Prima di eseguire:** `Runtime → Change runtime type → T4 GPU`.
**Secrets** (icona chiave a sinistra): aggiungi `GH_TOKEN` con un personal access
token GitHub che ha accesso al repo privato (`Contents: Read`). Facoltativo:
`HF_TOKEN` per scaricare il modello più in fretta.


In [ ]:
!nvidia-smi || echo "NESSUNA GPU: Runtime -> Change runtime type -> T4 GPU"


In [ ]:
#@title 1) Scarica il codice da GitHub
import os, subprocess

REPO_DIR = "/content/steerlab-colab"
REPO_URL = "github.com/gio3002/steerlab-colab.git"

# Se il repo e' PUBBLICO non serve nessun token: si scarica e basta.
# Se e' PRIVATO serve un token nei Secrets di Colab (nome GH_TOKEN).
token = None
try:
    from google.colab import userdata
    token = userdata.get("GH_TOKEN")
except Exception:
    pass

url = f"https://{token}@{REPO_URL}" if token else f"https://{REPO_URL}"

if not os.path.isdir(REPO_DIR):
    r = subprocess.run(["git", "clone", url, REPO_DIR],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("DOWNLOAD FALLITO.")
        print("Se il repo e' privato manca il token: aggiungilo nei Secrets")
        print("con nome GH_TOKEN e 'Notebook access' attivo, poi riprova.")
        print(r.stderr[-600:])
        raise SystemExit(1)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], capture_output=True, text=True)

os.chdir(REPO_DIR)
print("codice scaricato in:", os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-2"],
                     capture_output=True, text=True).stdout)


In [ ]:
#@title 2) Dipendenze, tunnel e PREFLIGHT (controlla tutto prima di partire)
import subprocess, sys, os, json

r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
                   capture_output=True, text=True)
if r.returncode != 0:
    print("AVVISO pip (non bloccante):", r.stderr[-400:])

subprocess.run(["wget", "-q",
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    "-O", "cloudflared"], capture_output=True, text=True)
if os.path.exists("cloudflared"):
    os.chmod("cloudflared", 0o755)

import torch, transformers
print("PREFLIGHT")
print("  torch", torch.__version__, "| transformers", transformers.__version__)
print("  GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NESSUNA")
try:
    import bitsandbytes as bnb
    print("  bitsandbytes", bnb.__version__, "-> 4bit disponibile")
except Exception:
    print("  bitsandbytes assente -> il motore usera' fp16/bf16 (normale, nessun errore)")
print("  cloudflared:", "ok" if os.path.exists("cloudflared") else "MANCANTE")


In [ ]:
#@title 3) Calibrazione (una volta sola, ~5-10 min su T4)
# Produce calibration/calibration.json e directions.npz (estremi + equalizzatore).
# Se esiste gia' una calibrazione per QUESTO modello, la salta: i "Run all"
# successivi diventano veloci.
import os, json, subprocess, sys

skip = False
cal = "calibration/calibration.json"
if os.path.exists(cal) and os.path.exists("calibration/directions.npz"):
    try:
        with open(cal, encoding="utf-8") as f:
            old = json.load(f).get("model_id", "")
        cfg = json.load(open("config.json", encoding="utf-8"))
        cur = cfg.get("model_dir") or cfg.get("model_id")
        skip = (old == cur)
    except Exception:
        skip = False

if skip:
    print("calibrazione gia' presente per", cur, "-> salto")
else:
    r = subprocess.run([sys.executable, "-u", "calibrate.py"])
    print("calibrate.py exit:", r.returncode)


In [ ]:
#@title 4) Avvia l'app, ottieni IL LINK e LA CHIAVE DI CONTROLLO
import subprocess, sys, time, re, os

# Libera un avvio precedente (Run all ripetuto), altrimenti la porta e' occupata.
subprocess.run(["pkill", "-f", "app.py --tunnel"], capture_output=True)
time.sleep(1)

LOG = "app.log"
open(LOG, "w").close()
proc = subprocess.Popen([sys.executable, "-u", "app.py", "--tunnel"],
                        stdout=open(LOG, "w"), stderr=subprocess.STDOUT)
url = key = None
for _ in range(90):
    time.sleep(1)
    txt = open(LOG, encoding="utf-8", errors="replace").read()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", txt)
    k = re.search(r"DEPLOY_KEY=([0-9a-f]+)", txt)
    if m: url = m.group(0)
    if k: key = k.group(1)
    if url and key: break
print("\n" + "="*70)
print("LINK DA CONDIVIDERE :", url or "(non pronto)")
print("CHIAVE DI CONTROLLO :", key or "(non trovata)")
print("="*70)
if not url:
    print("exit:", proc.poll()); print("--- app.log ---"); print(txt[-1500:])
print("Manda all'agente QUESTO LINK e QUESTA CHIAVE: da quel momento\n"
      "modifiche, test ed esperimenti li fa lui, senza altri click.")


In [ ]:
#@title 5) Carica il modello in memoria (una volta, ~1 min)
import requests, time
t0 = time.time()
r = requests.post("http://127.0.0.1:7860/api/model/load", timeout=1800).json()
print("ok:", r.get("ok"), "· secondi:", round(time.time()-t0, 1))
print(r.get("state") or r.get("error"))


In [ ]:
#@title 6) (facoltativo) Scarica la calibrazione per committarla
# Se hai calibrato su Colab e vuoi che il repo la contenga:
from google.colab import files
files.download("calibration/calibration.json")
files.download("calibration/directions.npz")


## Note

- Il link è pubblico e temporaneo: vive finché il runtime resta acceso.
- La cronologia delle chat sta nel browser di chi usa l'app, non sul server.
- Se il modello non entra in memoria, riduci `max_new` o passa a
  `Qwen/Qwen2.5-1.5B-Instruct` in `config.json`.
- Per fermare tutto: `Runtime → Disconnect and delete runtime`.
